# CareLens · Find nearby care and see why it matched

CareLens helps people in India search for a service near a city or PIN and compare facilities by Fit Score or distance. The default example is **Find dialysis near Hyderabad 500003**.

**Run all cells on Python 3.10+** for the offline demonstration. The notebook contains the project source and the published facility, postal, and website reachability snapshots. The example runs without API credentials, PostgreSQL, or a network connection.

| Section | What you can reproduce |
|---|---|
| Data preparation | Inspect the cleaning results and optionally rebuild from the original facility ZIP and postal CSV |
| Search | Resolve the location, filter by radius, and find relevant services and specialties |
| Ranking and explanation | Apply the repository's scoring functions and display the match, authority, reachability, and evidence behind the results |
| Application source | Inspect the packaged frontend, API, database schema, and verification code |

**Source revision** [`develop` · `a522036`](https://github.com/ningyues/cmu-datathon-virtue-healthcare/tree/a5220367173f4c0129ed829a9eed4085e7896c19)  
**Data** Virtue Foundation facility records and India Post directory. [Datathon data resources](https://dais-for-good-2026.devpost.com/resources)


## 1. Configure the demonstration

Change the query or radius to explore the published snapshot. The local parser supports common service-first queries for dialysis, hemodialysis, peritoneal dialysis, cataract surgery, and MRI. Set `STRUCTURED_INTENT` to the app's intent payload for other service terms.


In [1]:
SEARCH_QUERY = "Find dialysis near Hyderabad 500003"
SEARCH_RADIUS_KM = 30
SORT_BY = "fit"  # "fit" or "distance"
TOP_N = 8
DETAIL_RANK = 1
STRUCTURED_INTENT = None

# Optional full cleaning replay. With True, set paths to the original inputs.
REBUILD_FROM_RAW = False
RAW_DATATHON_ZIP = ""
RAW_POSTAL_CSV = ""


## 2. Load the bundled project

The payload cells contain the pinned repository source and data. The loader verifies the archive checksum and extracts it to a temporary directory for this session.


In [2]:
import base64
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

BUNDLE_SHA256 = "99557223fc2876457408986a60ef59e312d1b119b3424e0aa5226e578f3e8dd3"
_bundle_chunks = []


In [3]:
# Bundled source and data · part 1
_bundle_chunks.append("""
...
""")


In [4]:
# Bundled source and data · part 2
_bundle_chunks.append("""
...
""")


In [5]:
# Bundled source and data · part 3
_bundle_chunks.append("""
...
""")


In [6]:
# Bundled source and data · part 4
_bundle_chunks.append("""
...
""")


In [7]:
# Bundled source and data · part 5
_bundle_chunks.append("""
...
""")


In [8]:
# Bundled source and data · part 6
_bundle_chunks.append("""
...
""")


In [9]:
# Bundled source and data · part 7
_bundle_chunks.append("""
...
""")


In [10]:
# Bundled source and data · part 8
_bundle_chunks.append("""
...
""")


In [11]:
# Bundled source and data · part 9
_bundle_chunks.append("""
...
""")


In [12]:
# Bundled source and data · part 10
_bundle_chunks.append("""
...
""")


In [13]:
# Bundled source and data · part 11
_bundle_chunks.append("""
...
""")


In [14]:
# Bundled source and data · part 12
_bundle_chunks.append("""
...
""")


In [15]:
# Bundled source and data · part 13
_bundle_chunks.append("""
...
""")


In [16]:
import sys

if sys.version_info < (3, 10):
    raise RuntimeError("Use Python 3.10 or later.")
if "_carelens_workdir" not in globals():
    _carelens_workdir = TemporaryDirectory(prefix="carelens-project-")
workdir = Path(_carelens_workdir.name)
project = workdir / "project"
bundle_zip = workdir / "snapshot.zip"
checksum = sha256()
with bundle_zip.open("wb") as stream:
    for chunk in _bundle_chunks:
        data = base64.b64decode("".join(chunk.split()), validate=True)
        checksum.update(data)
        stream.write(data)
del _bundle_chunks, chunk, data
if checksum.hexdigest() != BUNDLE_SHA256:
    raise ValueError("The bundled project did not pass the checksum check.")
with ZipFile(bundle_zip) as package:
    for member in package.namelist():
        if not (project / member).resolve().is_relative_to(project.resolve()):
            raise ValueError("Unexpected archive member path")
    package.extractall(project)

if str(project) not in sys.path:
    sys.path.insert(0, str(project))
data_dir = project / "data/processed"
evidence_dir = project / "data/evidence"
print("Verified the bundled source and published data snapshot.")


Verified the bundled source and published data snapshot.


## 3. Bronze → Silver → Gold

| Stage | Included logic |
|---|---|
| Bronze | Parse the original facility CSV, source row numbers, and saved audit evidence |
| Silver | Validate identifiers and coordinates, deduplicate fields and records, reconcile PINs, and apply reviewed address, website, service, and display-name decisions |
| Gold | Publish searchable facility names and locations with the PIN index, claim evidence, correction logs, and review queue |

The default run uses the bundled published tables. The facility loader applies the **46 reviewed display-name corrections** when it reads the published CSV. The optional rebuild invokes the packaged production cleaning script and uses its cleaned facilities, postal data, and evidence for the rest of the notebook.


In [17]:
import csv
import gzip
import json
import shutil
import subprocess
from collections import defaultdict
from decimal import Decimal, ROUND_HALF_UP
from html import escape

from app.facility_data import read_facilities
from app.location_matching import LocationMatcher, rank_by_distance
from pipeline.facility_names import read_name_decisions

csv.field_size_limit(16_000_000)

def csv_records(path):
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8-sig", newline="") as stream:
        yield from csv.DictReader(stream)

def show_table(rows):
    if not rows:
        print("No rows to display.")
        return
    columns = list(rows[0])
    header = "".join(f"<th>{escape(str(key))}</th>" for key in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(row.get(key, '')))}</td>" for key in columns) + "</tr>"
        for row in rows
    )
    table = f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"
    if callable(globals().get("displayHTML")):
        displayHTML(table)
    else:
        try:
            from IPython.display import HTML, display
        except ImportError:
            print(json.dumps(rows, indent=2, ensure_ascii=False, default=str))
        else:
            display(HTML(table))

if REBUILD_FROM_RAW:
    raw_zip, raw_postal = (Path(value).expanduser().resolve()
                           for value in (RAW_DATATHON_ZIP, RAW_POSTAL_CSV))
    if not raw_zip.is_file() or not raw_postal.is_file():
        raise FileNotFoundError("Set RAW_DATATHON_ZIP and RAW_POSTAL_CSV to the original files.")
    fresh = workdir / "rebuild"
    subprocess.run(
        [sys.executable, str(project / "pipeline/run_clean.py"), "--zip", str(raw_zip),
         "--postal", str(raw_postal), "--out", str(fresh)],
        cwd=project, check=True, capture_output=False,
    )
    # The app reads compressed CSVs in both loaders.
    for stem in ("facilities_clean", "pincode_offices_clean"):
        with (fresh / f"{stem}.csv").open("rb") as source, gzip.open(fresh / f"{stem}.csv.gz", "wb") as target:
            shutil.copyfileobj(source, target)
    facility_path = fresh / "facilities_clean.csv.gz"
    postal_path = fresh / "pincode_offices_clean.csv.gz"
    evidence_dir = fresh
else:
    facility_path = data_dir / "facilities_clean.csv.xz"
    postal_path = data_dir / "pincode_offices_clean.csv.gz"
    evidence_dir = project / "data/evidence"

facilities = list(read_facilities(facility_path))
cleaning_summary = json.loads((evidence_dir / "summary.json").read_text(encoding="utf-8"))
assert len(facilities) == cleaning_summary["hospital_output_rows"]
assert len({row["unique_id"] for row in facilities}) == len(facilities)
show_table([
    {"Measure": "Facility records", "Count": f"{len(facilities):,}"},
    {"Measure": "Cleaned postal offices", "Count": f"{cleaning_summary['post']['output_rows']:,}"},
    {"Measure": "Unique PINs", "Count": f"{cleaning_summary['post']['unique_pins']:,}"},
    {"Measure": "Reviewed display-name decisions", "Count": len(read_name_decisions())},
    {"Measure": "Open review issues", "Count": f"{cleaning_summary['review_queue_rows']:,}"},
])


Measure,Count
Facility records,"9,989"
Cleaned postal offices,"165,625"
Unique PINs,"19,586"
Reviewed display-name decisions,46
Open review issues,"1,552"


## 4. Resolve the query and filter locations

A six-digit PIN resolves to an approximate origin using the bundled postal reference. The location matcher keeps facilities within the requested straight-line distance that pass its PIN, state, and coordinate screens.


In [18]:
from app.service_search import SERVICES, match_search_intent, parse_query, parse_search_intent

if STRUCTURED_INTENT is None:
    parsed = parse_query(SEARCH_QUERY)
    search_payload = {
        "location": {"city": parsed.city, "postcode": parsed.pincode},
        "specialities": list(SERVICES[parsed.service_id].specialties),
        "service": [parsed.service_id.replace("_", " ")],
        "range": SEARCH_RADIUS_KM,
    }
else:
    search_payload = STRUCTURED_INTENT
intent = parse_search_intent(search_payload)
query_address = ", ".join(part for part in (intent.city, intent.pincode) if part)
matcher = LocationMatcher(postal_path)
location = matcher.resolve(query_address)
if location.status != "resolved":
    raise ValueError(f"Could not resolve {query_address!r} ({location.status}). Check the city or PIN.")
located = matcher.match(facilities, latitude=location.latitude, longitude=location.longitude,
                        radius_km=intent.range)
candidates = located["candidates"]
print(f"Origin: {location.label} ({location.source}). Radius: {intent.range:g} km.")
print(f"Service terms: {', '.join(intent.service)}. Specialties: {', '.join(intent.specialities)}.")
print(f"Facilities inside radius that pass the location screen: {len(candidates):,}.")
print(f"Records across the snapshot without a rankable distance: {len(located['unlocated_candidates']):,}.")


Origin: 500003, Telangana (postal_pin). Radius: 30 km.
Service terms: dialysis. Specialties: nephrology.
Facilities inside radius that pass the location screen: 356.
Records across the snapshot without a rankable distance: 1,832.


## 5. Rank and explain the matches

A specialty match adds 10 points. Each distinct service can add two points per matched field in procedure, equipment, and capability. CareLens normalizes the total to a match percentage.

**Authority** = 1 − 0.30 × geographic conflict flag − 0.50 × (1 − reachable URL share)  
**Fit Score** = Authority × match percentage

The notebook calls the repository's scoring functions using the bundled website reachability results. A fresh snapshot uses the repository's default geographic flag of zero with a status of `not_checked` for the separate address verification job. This status is distinct from the postal and coordinate screen applied to nearby searches. A live deployment can use confirmed geographic conflicts persisted in PostgreSQL.


In [19]:
from agents.verify_website import website_candidates
from app.scoring_backend import scoring

saved_reachability = {
    row["unique_id"]: row for row in csv_records(data_dir / "website_reachability.csv.gz")
}

def row_array(row, field):
    return [value.replace("\0", "") for value in json.loads(row[field] or "[]")]

def stored_website_evidence(row):
    saved = saved_reachability.get(row["unique_id"])
    if saved is None:
        return None, {}
    # PostgreSQL hashes a JSONB array of the two text columns, before URL normalization.
    source_values = [row["officialWebsite"], row["websites"]]
    source_hash = sha256(json.dumps(source_values, ensure_ascii=False).encode("utf-8")).hexdigest()
    if source_hash != saved["website_inputs_sha256"]:
        return None, {}
    ratio = Decimal(saved["reachable_ratio"])
    if not ratio.is_finite() or not 0 <= ratio <= 1:
        raise ValueError("Invalid bundled URL reachability share")
    return ratio, json.loads(saved["reachable_by_url"])

def rounded_score(number):
    return None if number is None else number.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

def rank_candidates(candidates, intent, sort_by):
    if sort_by not in {"fit", "distance"}:
        raise ValueError('SORT_BY must be "fit" or "distance".')
    specialties, terms = scoring.normalize_query(intent.specialities, intent.service)
    claim_evidence = {row["unique_id"]: row.get("service_evidence")
                      for row in match_search_intent(candidates, intent)}
    ranked = []
    for row in candidates:
        hospital = {field: row_array(row, field) for field in scoring.SERVICE_FIELDS}
        specialty_match = any(scoring.matches_field(row_array(row, "specialties"), term)
                              for term in specialties)
        service_matches = {term: scoring.calculate_service_score(hospital, term) for term in terms}
        reachable_ratio, checks = stored_website_evidence(row)
        authority = scoring.website_authority(0, reachable_ratio)
        breakdown = scoring.score_breakdown(
            authority, specialty_match, len(specialties),
            [points for points, _ in service_matches.values()],
        )
        if breakdown["raw_score"] == 0:
            continue
        ranked.append({
            **row, **breakdown,
            "fit_score": rounded_score(breakdown["final_score"]),
            "match_pct": rounded_score(breakdown["match_score_100"]),
            "authority": authority,
            "reachable_ratio": reachable_ratio,
            "saved_url_checks": checks,
            "geography_check_status": "not_checked",
            "service_matches": service_matches,
            "service_evidence": claim_evidence.get(row["unique_id"]),
        })
    ranked = rank_by_distance(ranked)
    if sort_by == "fit":
        ranked.sort(key=lambda row: -row["fit_score"] if row["fit_score"] is not None
                    else Decimal("Infinity"))
    return ranked

def fmt_pct(value):
    return "Pending" if value is None else f"{100 * value:.1f}%"

ranked = rank_candidates(candidates, intent, SORT_BY)
print(f"{len(ranked)} facilities matched a specialty or service. Ordered by {SORT_BY}.")
show_table([
    {"Rank": index, "Facility": row["name_standardized"] or row["name"],
     "City": row["address_city"], "Distance (km)": f"{row['distance_km']:.2f}",
     "Fit Score": "Pending" if row["fit_score"] is None else str(row["fit_score"]),
     "Match": f"{row['match_pct']}%", "URL reachability": fmt_pct(row["reachable_ratio"]),
     "Evidence type": ("Dataset excerpt" if row["service_evidence"] else
                       "Service text match" if row["service_score"] else "Related specialty")}
    for index, row in enumerate(ranked[:TOP_N], 1)
])


94 facilities matched a specialty or service. Ordered by fit.


Rank,Facility,City,Distance (km),Fit Score,Match,URL reachability,Evidence type
1,Sathya Kidney Centre And Super Specialty Hospitals,Hyderabad,4.30,87.50,87.50%,100.0%,Service text match
2,Anupama Hospital,Hyderabad,10.81,87.23,100.00%,74.5%,Service text match
3,Sree Manju Hospitals - Hyderabad,Hyderabad,11.83,86.17,87.50%,97.0%,Service text match
4,ApoKos Rehabilitation Hospital,Hyderabad,9.16,82.25,87.50%,88.0%,Service text match
5,Preeti Urology and Kidney Hospital - KPHB Colony,Hyderabad,11.81,78.75,87.50%,80.0%,Dataset excerpt
6,Udai Omni Hospital Formrly Udai clinic Orthopedic Cntr Hyderabad,Hyderabad,5.50,78.57,87.50%,79.6%,Service text match
7,Kims Hospital,Hyderabad,5.05,77.88,87.50%,78.0%,Service text match
8,Xenia Hospitals,Hyderabad,8.79,76.92,100.00%,53.8%,Service text match


## 6. Inspect one result

The service score measures matches in the dataset. Website reachability records whether a link opened at the time of the saved check. The reviewed claims table provides source-backed statements where available. Use the packaged facility row for current postal and coordinate status.


In [20]:
if ranked and 1 <= DETAIL_RANK <= len(ranked):
    chosen = ranked[DETAIL_RANK - 1]
    print(chosen["name_standardized"] or chosen["name"])
    print(f"Fit Score: {chosen['fit_score'] if chosen['fit_score'] is not None else 'Pending'}")
    print(f"Authority: {fmt_pct(chosen['authority'])} · Geographic address verification: {chosen['geography_check_status']}")
    print(f"Postal screen: {chosen['pin_status']} · Coordinate screen: {chosen['geography_status']}")
    print(f"Match: {chosen['raw_score']} of {chosen['max_raw_score']} points "
          f"({chosen['specialty_score']} specialty, {chosen['service_score']} service).")
    show_table([
        {"Service": term, "Points": points, **{key.capitalize(): "Match" if value else "—"
                                                for key, value in matches.items()}}
        for term, (points, matches) in chosen["service_matches"].items()
    ])
    if match := chosen["service_evidence"]:
        print(f"Dataset {match['field']} evidence: {match['text']}")
    else:
        print("This result is based on dataset matches. A service excerpt passing the explicit evidence filter was unavailable.")
    score_excerpts = []
    for term, (_, matched_fields) in chosen["service_matches"].items():
        for field, matched in matched_fields.items():
            if matched:
                excerpt = next(value for value in row_array(chosen, field)
                               if scoring.matches_field([value], term))
                score_excerpts.append({"Service": term, "Field": field,
                                       "Dataset text matched by the scorer": excerpt[:250]})
    if score_excerpts:
        show_table(score_excerpts)
    checked = chosen["saved_url_checks"]
    print("Saved URL reachability examples")
    show_table([
        {"URL": url, "Reachability": (
            "Opened" if checked.get(url) is True else
            "Unavailable at check" if checked.get(url) is False else "Unchecked")}
        for url in website_candidates(chosen)[:5]
    ])
    reviewed = [
        {"Claim": row["claim"], "Status": row["status"],
         "Source": row["source_url"], "Reviewed": row["reviewed_on"]}
        for row in csv_records(evidence_dir / "verified_service_claims.csv")
        if row["unique_id"] == chosen["unique_id"]
    ]
    if reviewed:
        print("Reviewed service or specialty claims for this facility")
        show_table(reviewed)
    else:
        print("The bundled review table has no source-reviewed claims for this facility.")
else:
    print("Choose an available rank to see a facility's evidence.")


Sathya Kidney Centre And Super Specialty Hospitals
Fit Score: 87.50
Authority: 100.0% · Geographic address verification: not_checked
Postal screen: state_consistent · Coordinate screen: postal_distance_within_50_km
Match: 14 of 16 points (10 specialty, 4 service).
This result is based on dataset matches. A service excerpt passing the explicit evidence filter was unavailable.
Saved URL reachability examples
The bundled review table has no source-reviewed claims for this facility.


Service,Points,Capability,Equipment,Procedure
dialysis,4,—,Match,Match


Service,Field,Dataset text matched by the scorer
dialysis,equipment,Has 20 haemodialysis machines
dialysis,procedure,Haemodialysis using dialysis machines in Nephrology


URL,Reachability
https://sathyakidneycentre.com/,Opened
https://scclmines.com/obms/General/gHospList.aspx,Opened
https://www.nirujahealthtech.com/15-best-hospitals-in-hyderabad/,Opened
https://sathyakidneycentre.com/doctors/,Opened
https://www.facebook.com/104049135781409,Opened


## 7. Explore the full implementation

The app uses React, Vite and Leaflet for the interface, with a Python search API and PostgreSQL scoring backend. The bundled repository includes the production search API, database schema and migrations, live verification modules, frontend and tests.

| Project area | Source |
|---|---|
| Cleaning implementation and reviewed decisions | [`pipeline/`](https://github.com/ningyues/cmu-datathon-virtue-healthcare/tree/a5220367173f4c0129ed829a9eed4085e7896c19/pipeline) |
| Search, location and API | [`app/`](https://github.com/ningyues/cmu-datathon-virtue-healthcare/tree/a5220367173f4c0129ed829a9eed4085e7896c19/app) |
| Ranking and persistence | [`virtue-healthcare/`](https://github.com/ningyues/cmu-datathon-virtue-healthcare/tree/a5220367173f4c0129ed829a9eed4085e7896c19/virtue-healthcare) |
| Link, accreditation and rating integrations | [`agents/`](https://github.com/ningyues/cmu-datathon-virtue-healthcare/tree/a5220367173f4c0129ed829a9eed4085e7896c19/agents) |
| Web interface | [`frontend/`](https://github.com/ningyues/cmu-datathon-virtue-healthcare/tree/a5220367173f4c0129ed829a9eed4085e7896c19/frontend) |

The notebook's example is a reproducible offline walkthrough. The production instructions in the bundled `README.md` cover PostgreSQL, Ark, and the additional live evidence integrations.


In [21]:
def show_source(relative_path):
    source = (project / relative_path).resolve()
    if not source.is_relative_to(project.resolve()) or not source.is_file():
        raise FileNotFoundError(relative_path)
    print(source.read_text(encoding="utf-8"))

# Examples: show_source("pipeline/run_clean.py") or show_source("app/api.py")
print("Project source extracted. Use show_source(relative_path) to inspect any text file.")


Project source extracted. Use show_source(relative_path) to inspect any text file.
